In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
import pandas as pd

lst_mthend = '2025-06-30'

In [0]:
# Load AVY tNPS customer list
df = spark.read.format("csv").load(
    "/mnt/lab/vn/project/scratch/adhoc/adhoc_avy_tnps_analysis/csv/avy_tnps_202506.csv",
    header=True,
    inferSchema=True
).withColumn(
    "tnps",
    F.when(F.col("score") <= 6, "Detractor")
    .when(F.col("score") <= 8, "Indifference")
    .otherwise("Promoter")
).withColumn(
    "image_date", 
    F.last_day(F.add_months(F.col("local_dt"), -1))
)

# Create a window specification to order by local_dt
window_spec = Window.partitionBy("cli_num").orderBy(F.col("local_dt").desc())

# Add a row number to each row within the cli_num partition
df_with_row_num = df.withColumn("row_num", F.row_number().over(window_spec))

# Cache the DataFrame with row numbers
df_with_row_num.cache()

# Filter to get the latest record for each cli_num
avy_dedup_df = df_with_row_num.filter(F.col("row_num") == 1).drop("row_num")

# Filter to get the duplicated records and select the lowest row_num for each cli_num
dup_df = df_with_row_num.filter(
    F.col("row_num") > 1
).withColumn(
    "row_num_dup", 
    F.row_number().over(Window.partitionBy("cli_num").orderBy(F.col("row_num")))
).filter(
    F.col("row_num_dup") == 1
).select(
    F.col("cli_num"),
    F.col("score").alias("score_before"),
    F.col("tnps").alias("tnps_before"),
    F.col("local_dt").alias("local_dt_before")
)

# Join the duplicated records back to the unique records
avy_df = avy_dedup_df.join(
    dup_df, 
    on="cli_num", 
    how="left"
)

# Load Customer datamart to get Demographic information
custdm_cols = [
    "cli_num", "sex_code", "cur_age", "occp_desc", "image_date"
]
custdm = spark.read.format("parquet").load(
    "/mnt/prod/Curated/VN/Master/VN_CURATED_DATAMART_DB/TCUSTDM_MTHEND/"
).select(
    custdm_cols
).join(
    avy_df,
    on=["image_date", "cli_num"],
    how="inner"
)

# Load COSMO data
cosmo_cols = [
    "userId", "userIdType", "transactionType", "eventType", "eventTimestamp",
    "responseStatus"
    
]
cosmo_path = "/mnt/prod/Published/VN/Full/VN_PUBLISHED_EXTERNAL_DB/VN_DECREE53_CWS/*/*/*/*.parquet"
cosmo_df = spark.read.format("parquet").load(
    cosmo_path
).filter(
    (F.col("transactionType") == "login") &
    (F.col("eventType") == "authenticate") &
    (F.col("responseStatus") == 200)
).select(
    cosmo_cols
).withColumn(
    "login_date",
    F.to_date(F.from_utc_timestamp(F.col("eventTimestamp"),'UTC+8'),"yyyy-MM-dd'T'HH:mm:ss")
).drop(
    "eventTimestamp"
)


In [0]:
# print(avy_df.count())
# check_dup(avy_df, "cli_num")

### 4_Load metrics for analysis
CWS activity<br>
MOB / Tenure<br>
VIP status<br>
Product holding<br>
Demographic<br>
Agent segment / Tier<br>

In [0]:
# Derive AVY customers who logged on CWS before survey
cws_login_df = spark.sql("""
SELECT  DISTINCT
        a.cli_num,
        'Y' AS cws_login_ind
FROM    {avy} a
    INNER JOIN
        vn_published_sfdc_easyclaims_db.Account acc ON a.cli_num = acc.INTEGRATION_KEY__C
    INNER JOIN
        {cosmo} c ON c.userId = acc.MCF_User_Id__pc
WHERE   a.local_dt > c.login_date
""",
avy = avy_df,
cosmo = cosmo_df)

# print(cws_login_df.count())
# check_dup(cws_login_df, "cli_num")

In [0]:
# Finding desire base (as of latest date)
inf_cus_base = spark.sql("""
with prod_list as (
select  PLAN_CODE, INS_TYP
        , case when fld.FLD_VALU_DESC='Investment' then
                case when pln.PLAN_CODE like 'RUV%' then 'Investment-ILP'
                    when pln.PLAN_CODE like 'UL%' then 'Investment-UL'
                    else 'Investment-Others' end
               when fld.FLD_VALU_DESC='Whole Life' then
                case when pln.PLAN_CODE not like '%ED99' then 'Endowment'
                    else 'Whole Life' end
               when pln.PLAN_CODE in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
               when pln.PLAN_CODE in ('BHC9I','CA360','CI360','CN360','CX360','PN001') then 'Micro-Activator'
              else fld.FLD_VALU_DESC end
         as PROD_TYPE
from    vn_published_cas_db.tplans pln inner join
        vn_published_cas_db.tfield_values fld on pln.INS_TYP = fld.FLD_VALU and fld.FLD_NM='INS_TYP'
where   1=1
  and   pln.CVG_TYP = 'B'
), vip_status as (
select  CLI_NUM,
        case 
            when VIP_TYP_DESC = 'SUPER VIP' then '1. Super VIP'
            when VIP_TYP_DESC = 'VIP Platinum' then '2. VIP Platinum'
            when VIP_TYP_DESC = 'VIP Gold' then '3. VIP Gold'
            when VIP_TYP_DESC = 'VIP Silver' then '4. VIP Silver'
        end as VIP_STATUS,
        image_date
from    vn_published_casm_cas_snapshot_db.twrk_client_ape
), inf_cus as (
select  -- holding RUV
        distinct cpl.CLI_NUM
        , count(DISTINCT pol.POL_NUM) as POL_COUNT
        , collect_set(pln.PROD_TYPE) as LIST_OF_PRODUCT_TYPES
        , collect_set(rpl.PROD_TYP) as LIST_OF_RIDER_TYPES
        --, MAX(nvl(vip.VIP_STATUS, '5. NonVIP')) as VIP_STATUS
from    vn_published_casm_cas_snapshot_db.tpolicys pol
    inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on pol.POL_NUM = cpl.POL_NUM and cpl.LINK_TYP='O' and cpl.REC_STATUS='A' and pol.image_date = cpl.image_date
    inner join
        vn_published_casm_ams_snapshot_db.tams_agents agt on pol.AGT_CODE = agt.AGT_CODE and pol.image_date = agt.image_date
    left join
        vn_published_casm_cas_snapshot_db.tcoverages rid on pol.POL_NUM = rid.POL_NUM and rid.CVG_TYP='R' and rid.CVG_STAT_CD in ('1','2','3','5','7','9') and pol.image_date = rid.image_date
    left join 
        prod_list pln on pol.PLAN_CODE_BASE = pln.PLAN_CODE 
    left join
        vn_published_cas_db.tplans rpl on rid.PLAN_CODE = rpl.PLAN_CODE
    left join
        vip_status vip on cpl.CLI_NUM = vip.CLI_NUM and cpl.image_date = vip.image_date
where     1=1
    and     pol.POL_STAT_CD in ('1','3','5')                            -- in-force or Maturing 
group by  cpl.CLI_NUM
)
select    *
from      inf_cus
""")

# Step 1: Create indicator columns for product types
port_holding_df = inf_cus_base.withColumn(
    "ILP_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-ILP"), "Y").otherwise("N")
).withColumn(
    "UL_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-UL"), "Y").otherwise("N")
).withColumn(
    "ENDOWMENT_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Endowment"), "Y").otherwise("N")
).withColumn(
    "WHOLE_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Whole Life"), "Y").otherwise("N")
).withColumn(
    "TERM_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Term Life"), "Y").otherwise("N")
).withColumn(
    "DIGITAL_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Digital"), "Y").otherwise("N")
).withColumn(
    "MICRO_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Micro-Activator"), "Y").otherwise("N")
).withColumn(
    "OTH_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Accident"), "Y").otherwise("N")
)

# Step 2: Explode LIST_OF_RIDER_TYPES into dynamic columns
unique_rider_types = port_holding_df.select(F.explode(F.col("LIST_OF_RIDER_TYPES")).alias("rider")).distinct().collect()
rider_type_columns = [rider.rider for rider in unique_rider_types]

for rider_type in rider_type_columns:
    port_holding_df = port_holding_df.withColumn(
        f"{rider_type}_IND",
        F.when(F.array_contains(F.col("LIST_OF_RIDER_TYPES"), rider_type), "Y").otherwise("N")
    )

port_holding_df = port_holding_df.drop('LIST_OF_PRODUCT_TYPES','LIST_OF_RIDER_TYPES')
# print(port_holding_df.count())

In [0]:
# Add `rider_ind` and `product_type` flag
new_cols = ({
    'rider_ind': F.greatest(F.col('cancer_rider_ind'),
                            F.col('add_rider_ind'),
                            F.col('tpd_rider_ind'),
                            F.col('tp_rider_ind'),
                            F.col('mc_rider_ind'),
                            F.col('ci_rider_ind'),
                            F.col('hc_rider_ind')),

    'product_type': F.when((F.col('ilp_ind') == 'Y') &
                           (F.greatest(F.col('ul_ind'),
                                     F.col('endowment_ind'),
                                     F.col('whole_ind'),
                                     F.col('term_ind'),
                                     F.col('digital_ind'),
                                     F.col('micro_ind'),
                                     F.col('oth_ind')) == 'N'),
                           'ILP only')
                    .when((F.col('ul_ind') == 'Y') &
                           (F.greatest(F.col('ilp_ind'),
                                     F.col('endowment_ind'),
                                       F.col('whole_ind'),
                                       F.col('term_ind'),
                                       F.col('digital_ind'),
                                       F.col('micro_ind'),
                                       F.col('oth_ind')) == 'N'),
                           'UL only')
                    .when((F.col('endowment_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Endowment only')
                    .when((F.col('whole_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Whole Life only')
                    .when((F.col('term_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Term Life only')
                    .when((F.col('digital_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Digital only')
                    .when((F.col('micro_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Micro only')
                    .when((F.col('oth_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                      F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind')) == 'N'),
                          'Accident only')
                    .when(F.greatest(F.col('ilp_ind'),
                                    F.col('ul_ind'),
                                    F.col('endowment_ind'),
                                    F.col('whole_ind'),
                                    F.col('term_ind'),
                                    F.col('digital_ind'),
                                    F.col('micro_ind'),
                                    F.col('oth_ind')) == 'Y', 'Multiple products')
})

port_holding_df = port_holding_df.withColumns(
      new_cols
).withColumn(
      "pol_count_cat",
      F.when(F.col("POL_COUNT") == 1, "1. Single Policy")
      .when(F.col("POL_COUNT") <= 3, "2. 2 - 3 Policies")
      .when(F.col("POL_COUNT") <= 5, "3. 4 - 5 Policies")
      .otherwise("4. >5 Policies")
)
port_holding_df = port_holding_df.toDF(*[col.lower() for col in port_holding_df.columns])

In [0]:
tenure_df = spark.sql(f"""
WITH tenure AS (
SELECT  po_num, client_tenure,
        CASE 
            WHEN client_tenure < 1 THEN '1. < 1yr'
            WHEN client_tenure < 2 THEN '2. 1 - 2yr'
            WHEN client_tenure < 5 THEN '3. 3 - 5yr'
            WHEN client_tenure <10 THEN '4. 6 - 10yr'
            ELSE '5. > 10yr'
        END AS tenure_cat,
        dpnd_spouse_ind,
        dpnd_child_ind,
        image_date
FROM    vn_curated_analytics_db.income_based_decile_agency
--WHERE   image_date = '{lst_mthend}'
UNION
SELECT  po_num, client_tenure,
        CASE 
            WHEN client_tenure < 1 THEN '1. < 1yr'
            WHEN client_tenure < 2 THEN '2. 1 - 2yr'
            WHEN client_tenure < 5 THEN '3. 3 - 5yr'
            WHEN client_tenure <10 THEN '4. 6 - 10yr'
            ELSE '5. > 10yr'
        END AS tenure_cat,
        dpnd_spouse_ind,
        dpnd_child_ind,
        image_date
FROM    vn_curated_analytics_db.income_based_decile_pd
--WHERE   image_date = '{lst_mthend}'
)
SELECT  image_date, po_num, MAX(tenure_cat) AS tenure_cat, MAX(dpnd_spouse_ind) AS dpnd_spouse_ind,
        MAX(dpnd_child_ind) AS dpnd_child_ind
FROM    tenure                  
GROUP BY
        image_date, po_num
""")

demo_df = spark.sql("""
SELECT  cli_num,    
        -- Demographic (Gender, Age, Marital status, Occupation)
        CASE WHEN cli.sex_code = 'F' THEN 'Female'
                ELSE 'Male'
        END                     AS gender,
        CASE WHEN cli.cur_age < 18 THEN 'a.<18'
                WHEN cli.cur_age BETWEEN 18 AND 29 THEN 'b.18 - 29'
                WHEN cli.cur_age BETWEEN 30 AND 34 THEN 'c.30 - 34'
                WHEN cli.cur_age BETWEEN 35 AND 44 THEN 'd.35 - 44'
                WHEN cli.cur_age BETWEEN 45 AND 54 THEN 'e.45 - 54'
                ELSE 'f.55+'
        END                     AS po_age,
        CASE WHEN cli.cur_age >= 18 AND seg.dpnd_child_ind > 0 THEN 'a.Married w/ kid'
                WHEN cli.cur_age >= 18 AND seg.dpnd_spouse_ind > 0 THEN 'b.Married'
                ELSE 'c.Unknown'
        END                     AS marital_status,

        -- Split OCCP_DESC into category and subcategory
        CASE 
            WHEN cli.occp_desc LIKE '%weler – Dealer' THEN 'Weler'
            WHEN cli.occp_desc LIKE '% - %' OR cli.occp_desc LIKE '% – %'
            THEN SUBSTRING(cli.occp_desc, 1, INSTR(cli.occp_desc, ' - ') - 1) 
            WHEN cli.occp_desc LIKE '%–%' OR cli.occp_desc LIKE '%–%'
            THEN SUBSTRING(cli.occp_desc, 1, INSTR(cli.occp_desc, '-') - 1)
            ELSE cli.occp_desc 
        END                     AS occp_cat,
        CASE 
            WHEN cli.occp_desc LIKE '%weler – Dealer' THEN 'Dealer'
            WHEN cli.occp_desc LIKE '% - %' OR cli.occp_desc LIKE '% – %'
            THEN SUBSTRING(cli.occp_desc, INSTR(cli.occp_desc, ' - ') + 3)
            WHEN cli.occp_desc LIKE '%–%' OR cli.occp_desc LIKE '%–%'
            THEN SUBSTRING(cli.occp_desc, 1, INSTR(cli.occp_desc, '-') + 2)
            ELSE NULL 
        END                     AS occp_sub_cat,
        NVL(seg.tenure_cat, '6. Missing') AS tenure_cat
FROM    {custdm} cli
    LEFT JOIN
        {tenure} seg ON cli.CLI_NUM = seg.po_num AND cli.image_date = seg.image_date
""",
tenure = tenure_df,
custdm = custdm)

# print(demo_df.count())
# check_dup(demo_df, "CLI_NUM")

In [0]:
# Load latest Agency Structure from MIS
loc_path = f"/mnt/lab/vn/project/scratch/adhoc/loc_to_psm_mapping_202505.xlsx"
sheet_name = "Structure"
cell_pos = "A1"
loc_to_psm = spark.read.format(
    "com.crealytics.spark.excel"
).option(
    "dataAddress", f"{sheet_name}!{cell_pos}"
).option(
    "header", "true"
).option(
    "treatEmptyValuesAsNulls", "true"
).option(
    "inferSchema", "true"
).option(
    "addColorColumns", "False"
).load(
    loc_path
).select(
    "loc_cd", "psm_name_8", "psm_name_9"
)

# Derive Agent status and Group
mpro_title = spark.sql(f"""
SELECT  agt.AGT_CODE AS agt_code,
        CASE WHEN stat_cd = '01' THEN 'INFORCE'
             WHEN stat_cd = '99' THEN 'TERMINATED'
        END AS agt_status,
        CASE WHEN trmn_dt IS NOT NULL THEN '4.Terminated'
             WHEN trmn_dt IS NULL THEN
                 CASE
                     WHEN comp_prvd_num IN ('01','02','08','96') THEN '1.Inforce'
                     WHEN comp_prvd_num = '04' THEN '2.CSC'
                     WHEN comp_prvd_num IN ('97','98') THEN '3.SM'
                     ELSE '5.Not-Agency'
                 END
        END AS AGT_RLTNSHP,
        COALESCE(
            CASE WHEN agt.MPRO_TITLE IS NOT NULL THEN
                CASE WHEN MDRT_DESC IN ('TOT', 'COT') THEN 'MDRT'
                     ELSE MDRT_DESC
                END
                ELSE 'Normal'
            END,
            MPRO_TITLE
        ) AS mpro_title,
        agt.LOC_CODE AS loc_cd,
        agt.image_date
FROM    vn_published_casm_ams_snapshot_db.tams_agents agt
WHERE   1=1
    AND agt.image_date >= '2024-12-31'
""")


par_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/"
).select(
    "agt_cd", "last_9m_pol", "last_3m_pol", "last_mth_pol", "image_date"
).withColumnRenamed(
    "agt_cd", "agt_code"
)

mpro_title = mpro_title.join(
    par_df, 
    on=["agt_code", "image_date"],
    how="left"
).join(
    loc_to_psm,
    "loc_cd",
    "left"
)

agt_df = mpro_title.withColumn(
    "agt_actvness",
    F.when(F.col("last_mth_pol") > 0, "1m active")
    .when(F.col("last_3m_pol") > 0, "3m active")
    .when(F.col("last_9m_pol") > 0, "9m active")
    .otherwise("SA")
).withColumn(
    "agt_segment",
    F.when(F.col("agt_actvness") != "SA",
        F.when(F.col("mpro_title") != "Normal", F.col("mpro_title"))
        .otherwise(F.col("agt_actvness"))
    ).otherwise(
        F.when(F.col("agt_rltnshp").isin("2.CSC","3.SM","4.Terminated"), "UCM")
        .otherwise("SA")
    )
).drop(
    "last_3m_pol", "last_9m_pol", "last_mth_pol", "agt_status", "AGT_RLTNSHP", "agt_actvness"
)

###5_Merge all metrics to Customer list

In [0]:
result = avy_df.join(
    cws_login_df,
    "cli_num",
    "left"
).join(
    port_holding_df,
    "cli_num",
    "left"
).join(
    demo_df,
    "cli_num",
    "left"
).join(
    agt_df,
    on=["agt_code", "image_date"],
    how="left"
).fillna(
    {
        "cws_login_ind": "N",
        #"vip_status": "5. NonVIP",
        "product_type": "Lapsed/Surr/Expired"
    }
)

# print(result.count())

In [0]:
result_pd = result.toPandas()

result_pd.to_csv(
    "/dbfs/mnt/lab/vn/project/scratch/adhoc/adhoc_avy_tnps_analysis/csv/result.csv",
    index=False,
    header=True,
    encoding='utf-8-sig'
)